In [10]:
import shutil, subprocess, tempfile
from pathlib import Path

# --- Locate src/data from a notebook in notebooks/ ---
NB_DIR = Path.cwd().resolve()
SRC_DIR = NB_DIR.parent / "src"
DATA_DIR = SRC_DIR / "data"

print("Notebook CWD:", NB_DIR)
print("SRC_DIR     :", SRC_DIR)
print("DATA_DIR    :", DATA_DIR)

if not SRC_DIR.exists() or not DATA_DIR.exists():
    raise FileNotFoundError(
        "Could not find sibling src/ and src/data/ next to notebooks/."
    )

# --- Mapping files to test individually ---
mapping_files = [
    "bfo-mapping.ttl",
    "ies-mapping.ttl",
    "ccom-mapping.ttl",
    "qudt-mapping.ttl",
    "ccot-mapping.ttl",
    "to-mapping.ttl",
    "time-mapping.ttl",
]

# --- Sanity: is ROBOT available? ---
ROBOT = shutil.which("robot")
if ROBOT is None:
    raise RuntimeError(
        "ROBOT not found on PATH. Install ROBOT and make sure 'robot' is runnable."
    )

subprocess.run([ROBOT, "--version"], check=False)

reasoners = ["HermiT", "ELK"]


def run_robot_reason(input_file: Path, reasoner: str) -> bool:
    """Run ROBOT on one mapping file; return True on success."""
    with tempfile.TemporaryDirectory() as temp_dir:
        output_file = Path(temp_dir) / "reasoned.ttl"

        res = subprocess.run(
            [
                ROBOT, "reason",
                "--reasoner", reasoner,
                "--input", str(input_file),
                "--output", str(output_file),
            ],
            capture_output=True,
            text=True,
        )

        if res.returncode != 0:
            print(res.stdout)
            print(res.stderr)

    return res.returncode == 0


any_ran = False

for name in mapping_files:
    path = DATA_DIR / name

    if not path.exists():
        print(f"SKIP (missing): {name}")
        continue

    any_ran = True

    for r in reasoners:
        ok = run_robot_reason(path, r)
        status = "SUCCESS" if ok else "FAILURE"
        print(f"{status}: {name} [{r}]")

if not any_ran:
    print("No mapping files found in src/data to test.")

Notebook CWD: C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\notebooks
SRC_DIR     : C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src
DATA_DIR    : C:\Users\kygal\repos\Ontology-Tradecraft\projects\project-3\assignment\src\data
SUCCESS: bfo-mapping.ttl [HermiT]
SUCCESS: bfo-mapping.ttl [ELK]
SKIP (missing): ies-mapping.ttl
SKIP (missing): ccom-mapping.ttl
SKIP (missing): qudt-mapping.ttl
SKIP (missing): ccot-mapping.ttl
SKIP (missing): to-mapping.ttl
SKIP (missing): time-mapping.ttl
